# 🏗️ Notebook 1: Collaborative Whiteboard — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎨 What we're designing

A real-time collaborative whiteboard — think **Miro**, **Figma (FigJam)**, **Excalidraw**, **tldraw**, or **Google Jamboard**.

Many people draw, move, and delete shapes on the same canvas *at the same time*. Every edit should appear on everyone else's screen within a blink (~100 ms), and in the end everyone must see **exactly the same canvas**, even if their network dropped for a minute.

> **Beginner mental model:** imagine a shared Google Doc, but instead of typing letters, users drag rectangles and scribble lines. The same ideas apply (edits merge, late joiners see current state, offline edits reconcile), but the *unit of edit* is a shape, not a character.


## ✅ Requirements

### Functional
- Draw / move / delete shapes in real time.
- See other users' **cursors and selections** (presence).
- **Late joiners** load the current state of the board.
- **Offline edits** merge back correctly when the user reconnects.
- **Undo / redo** per user.

### Non-functional
- **Convergence** — all clients end up with the *same* canvas (eventual consistency).
- **Fan-out latency** — an edit reaches other users in < 100 ms p50.
- **Availability** — tolerate brief network partitions; never show a blank or wrong canvas.
- **Scalability** — 100k+ concurrent boards, 1M+ open WebSocket connections.
- **Durability** — boards survive server restarts; don't lose strokes.


## 🔢 Back-of-envelope (runnable)

Numbers are intentionally round so they are easy to reason about. The one that matters is
**egress**, and it is the one people get wrong: a whiteboard is a *fan-out* system, so every
byte a user sends leaves the building multiplied by the number of people in the room.

In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────
concurrent_boards   = 100_000
users_per_board     = 5
edits_per_sec_board = 10          # a drag emits many small ops
bytes_per_edit      = 300         # id + coords + colour + lamport + actor
duty_cycle          = 0.20        # fraction of boards where someone is actively drawing
cursor_hz           = 20          # presence updates per user per second while moving
bytes_per_cursor    = 60
conns_per_ws_node   = 50_000

# ── Connections ───────────────────────────────────────────────────────────
connections = concurrent_boards * users_per_board
ws_nodes    = connections / conns_per_ws_node

# ── Ops: ingress vs egress ────────────────────────────────────────────────
# INGRESS = what clients send us.
edits_per_sec = concurrent_boards * edits_per_sec_board * duty_cycle
ingress_mb_s  = edits_per_sec * bytes_per_edit / 1e6

# EGRESS = what we send back out. Each edit goes to the OTHER users on that board.
# THIS is the number that sizes the fan-out tier, and it is ~4x the ingress.
fanout        = users_per_board - 1
egress_mb_s   = edits_per_sec * fanout * bytes_per_edit / 1e6

# Presence is small per message but very high frequency -- check it separately,
# because "it's only cursors" is how people accidentally 3x their bandwidth bill.
cursor_msgs_s   = connections * cursor_hz * duty_cycle
cursor_egress_mb_s = cursor_msgs_s * fanout * bytes_per_cursor / 1e6

total_egress_mb_s = egress_mb_s + cursor_egress_mb_s

# ── Durability: the op log ────────────────────────────────────────────────
oplog_tb_per_day = ingress_mb_s * 86_400 / 1e6

print(f"connections            {connections:>12,}   -> {ws_nodes:>5,.0f} WS nodes")
print()
print(f"edits/sec              {edits_per_sec:>12,.0f}")
print(f"ops  INGRESS           {ingress_mb_s:>12,.0f} MB/s")
print(f"ops  EGRESS            {egress_mb_s:>12,.0f} MB/s   (x{fanout} fan-out)")
print(f"cursor EGRESS          {cursor_egress_mb_s:>12,.0f} MB/s")
print(f"TOTAL egress           {total_egress_mb_s:>12,.0f} MB/s = {total_egress_mb_s*8/1000:,.0f} Gbps")
print(f"  per WS node          {total_egress_mb_s/ws_nodes*8:>12,.0f} Mbps")
print()
print(f"op log growth          {oplog_tb_per_day:>12,.1f} TB/day  <- why snapshots + truncation exist")

### What the numbers force us to do

1. **Egress is ~4× ingress, and cursors dominate it.** Presence is a handful of bytes per
   message but fires 20×/second per user, so it can outweigh the actual drawing. Fixes:
   coalesce cursor updates to ~10 Hz server-side, send deltas not absolutes, and drop
   (never queue) stale cursor positions — the newest position makes older ones worthless.
2. **A single WS node has to push ~0.6 Gbps to 50k sockets.** That is fine on a modern
   NIC, but it means the fan-out loop
   must not do per-recipient serialisation — serialise the frame once, `send` the same
   bytes N times.
3. **The op log grows terabytes per day** at these rates. You cannot keep every op forever:
   snapshot periodically, then truncate ops older than the snapshot. That is the whole
   justification for §5 of notebook 3.

⚖️ **Where this is soft:** `duty_cycle = 0.20` is a guess and it scales *everything*
linearly. `users_per_board = 5` is an average that hides the tail — a 200-person all-hands
board has a fan-out of 199, and that single board can cost as much as 50 normal ones. Design
the fan-out path for the tail, then size the fleet with the average.

## 🧱 High-level architecture

```
 ┌──────────┐   WebSocket    ┌─────────────┐    ┌───────────────┐
 │ Browser  │ ─────────────► │ WS Gateway  │──► │ Room Service  │
 └──────────┘                │ (stateless) │    │  (per board)  │
                             └──────┬──────┘    └──────┬────────┘
                                    │                  │
                                    ▼                  ▼
                         ┌─────────────────┐   ┌────────────────┐
                         │ Redis Pub/Sub   │   │ CRDT / Op Log  │
                         │ (board:<id>)    │   │   (per board)  │
                         └────────┬────────┘   └──────┬─────────┘
                                  │                   │
                                  ▼                   ▼
                         other WS Gateways     Snapshot store (S3)
                         (cross-node fan-out)   taken every ~1k ops
```

### Responsibilities
- **WS Gateway** — terminates WebSockets. Stateless so we can autoscale. Just a relay.
- **Room Service** — validates ops for a board, assigns a Lamport clock, writes to the op log, publishes to pub/sub.
- **Redis Pub/Sub** — broadcasts ops to every gateway that has at least one subscriber on that board.
- **CRDT / Op Log** — durable per-board store. Replaying ops reconstructs the board.
- **Snapshot store** — periodic full state so new joiners don't replay millions of ops.


## 🏚️ → 🏛️ Bad → Better → Best architecture

A good interview answer **explains what you rejected and why**, not just the final picture.

### 🏚️ Naive v0 — single server, broadcast to all
- One process keeps an in-memory map `board_id → set[WebSocket]`.
- When a user sends an op, server pushes it to every other socket in that room.

**Why it fails:**
- Single point of failure — restart = everyone disconnects and loses in-flight ops.
- One box can't hold millions of sockets.
- No durability — reload the page and your whiteboard is gone.
- No offline merge — the server *is* the source of truth; if two users edit while partitioned, one wins arbitrarily (lost update).

### 🏗️ v1 — sharded WS gateways + pub/sub
- Add many WS gateways behind a load balancer.
- Use Redis Pub/Sub so any node can fan out to sockets on any other node.
- Persist ops to a database (append-only op log).

**Better, but still:**
- Pub/Sub is fire-and-forget — if a gateway misses a message during restart, clients desync.
- Late joiners still have to replay the whole op log.
- Concurrent edits still race unless we add logical clocks.

### 🏛️ v2 — CRDT + snapshots + resumable streams
- Model the board as a **CRDT** (we use a Last-Writer-Wins map in notebook 3). Concurrent edits merge deterministically — no central serialization needed.
- **Snapshots** every N ops → new joiners download snapshot + replay tail.
- Clients keep a **since-cursor** (last seen Lamport). On reconnect they ask "give me ops > cursor", and pub/sub is just an optimization on top of a durable op log.
- Presence (cursors) is **ephemeral** — pub/sub only, never persisted.

This is roughly what **Figma**, **Liveblocks**, **Yjs**, and **Automerge** converge on.


## ⚔️ The load-bearing decision: OT or CRDT?

Everything in notebooks 2 and 3 rests on this one choice, so we are going to *earn* it
rather than assert it. Both approaches solve the same problem — **two people edited at the
same time; produce one answer everyone agrees on** — and they solve it in opposite ways:

| | **OT** (Operational Transform) | **CRDT** (Conflict-free Replicated Data Type) |
|---|---|---|
| Idea | Rewrite the incoming op so it still means the right thing given what already happened | Design the data type so that merging is order-independent by construction |
| Needs a central server? | **Yes** in practice — someone must define "what already happened" | No — any two replicas can merge directly |
| Metadata per op | Tiny (an index) | Timestamps, actor ids, tombstones |
| Where the difficulty lives | In the transform functions | In the data model |

The first cell shows *why OT works*. The second shows *what it costs*. Then we pick.

### Step 1 — see the actual problem, and see OT solve it

Two people edit `"HELLO"` at the same moment. Alice appends ` WORLD` at index 5; Bob
prepends `OH ` at index 0. Each op was written against the *original* document, so when
Alice's op arrives at Bob's replica, its index 5 is already stale — Bob's insert shifted
everything right by 3.

In [ ]:
# A deliberately tiny OT for a text buffer. Ops: ("ins", index, text, actor) / ("del", index, actor)
def apply_op(doc: str, op) -> str:
    if op[0] == "noop": return doc
    if op[0] == "ins":  return doc[:op[1]] + op[2] + doc[op[1]:]
    return doc[:op[1]] + doc[op[1] + 1:]

def transform(a, b):
    """T(a, b): rewrite op `a` so it is correct to apply AFTER concurrent op `b`."""
    if a[0] == "ins" and b[0] == "ins":
        # Same index? Break the tie consistently or the two sites diverge.
        if a[1] < b[1] or (a[1] == b[1] and a[3] < b[3]):
            return a
        return ("ins", a[1] + len(b[2]), a[2], a[3])
    if a[0] == "ins" and b[0] == "del":
        return a if a[1] <= b[1] else ("ins", a[1] - 1, a[2], a[3])
    if a[0] == "del" and b[0] == "ins":
        return a if a[1] < b[1] else ("del", a[1] + len(b[2]), a[3])
    # del/del — the interesting one: both deleted the same character.
    if a[1] < b[1]: return a
    if a[1] > b[1]: return ("del", a[1] - 1, a[3])
    return ("noop", 0, a[3])          # already gone; do nothing

doc   = "HELLO"
alice = ("ins", 5, " WORLD", "alice")
bob   = ("ins", 0, "OH ",    "bob")

# ❌ Naive: just replay the other side's op as-is.
naive_at_alice = apply_op(apply_op(doc, alice), bob)
naive_at_bob   = apply_op(apply_op(doc, bob),   alice)
print("naive  @alice:", repr(naive_at_alice))
print("naive  @bob  :", repr(naive_at_bob))
print("converged?", naive_at_alice == naive_at_bob, "<- stale indices corrupt the document\n")

# ✅ OT: transform the remote op against the local one before applying it.
ot_at_alice = apply_op(apply_op(doc, alice), transform(bob,   alice))
ot_at_bob   = apply_op(apply_op(doc, bob),   transform(alice, bob))
print("OT     @alice:", repr(ot_at_alice))
print("OT     @bob  :", repr(ot_at_bob))
assert ot_at_alice == ot_at_bob
print("converged? True  <- and the result preserves BOTH intentions")

OT is genuinely good at this. Note what it preserved: not just *a* consistent answer, but
the answer a human would want — both insertions survive, in the right places. That property
is called **intention preservation**, and it is why Google Docs shipped OT for text.

### Step 2 — now count what OT costs on *our* data model

The transform function above handles a document with **2** operation types (`ins`, `del`),
so it needs a rule for each ordered pair. A whiteboard's op set is bigger, and every pair
needs a rule that is correct against every other — including the pairs nobody thinks about,
like "resize a shape" vs "delete the group that shape is in".

In [ ]:
text_ops  = ["insert", "delete"]
board_ops = ["add", "move", "resize", "recolor", "delete", "group", "reorder-z"]

def transform_rules(ops):
    return len(ops) ** 2          # T(a, b) for every ordered pair

print(f"text editor : {len(text_ops)} op types -> {transform_rules(text_ops):>2} transform rules")
print(f"whiteboard  : {len(board_ops)} op types -> {transform_rules(board_ops):>2} transform rules")
print(f"add ONE feature (e.g. 'lock shape'): {transform_rules(board_ops + ['lock'])} rules "
      f"(+{transform_rules(board_ops + ['lock']) - transform_rules(board_ops)})")
print()
print("Each rule must satisfy TP1 (two sites converge) and, for peer-to-peer OT,")
print("TP2 (three concurrent ops converge regardless of order). TP2 is where published")
print("OT algorithms have historically been found to be wrong, years after shipping.\n")

# The LWW-CRDT alternative: ONE merge rule, independent of how many op types exist.
def lww_merge(current, incoming):
    """Whole-shape last-writer-wins. `ts` = (lamport, actor) -- a stable total order."""
    return incoming if current is None or incoming["ts"] > current["ts"] else current

for n_ops in (2, 7, 20):
    print(f"{n_ops:>2} op types -> OT: {n_ops**2:>3} transform rules   |   LWW-CRDT: 1 merge rule")

### Step 3 — the decision, and what it costs us

**We choose a Last-Writer-Wins map CRDT** (`shape_id → (lamport, actor, shape)`), for three
reasons that are specific to *this* product:

1. **Our unit of edit is a whole shape, not a character in a sequence.** OT's superpower is
   merging two edits *inside one ordered sequence*. Shapes have no ordering that a user
   perceives — moving rect A and recolouring rect B are simply independent. The problem OT
   is brilliant at mostly does not arise here.
2. **Offline is a hard requirement** (see the functional list above). OT wants a server to
   define the canonical order; a client that has been on a train for an hour has no such
   order. CRDT merge works peer-to-peer, which makes offline a special case of normal
   operation rather than a separate code path.
3. **The op-type count grows with the product.** Every new tool a designer adds is another
   row *and column* in the transform matrix. One merge rule does not grow.

⚖️ **And here is what that choice costs us — say this part out loud, it is the honest half:**

| Cost | Detail |
|---|---|
| **Lost updates within a shape** | Alice drags a rect while Bob recolours it. Whole-shape LWW keeps one op and silently discards the other. Notebook 3 §3 shows this happening, and shows the per-field fix. |
| **Metadata per shape** | A lamport + actor on every shape (~16–24 bytes). At 300 B/op that is ~7% overhead, forever. |
| **Tombstones never fully die** | A deleted shape must leave a tombstone or a late `add` resurrects it. Tombstones need garbage collection coordinated across *all* replicas — which is a genuinely hard problem, usually solved by "GC anything older than the oldest live snapshot cursor". |
| **No intention preservation** | For the text tool on the canvas, two people typing in the same text box *will* clobber each other under shape-level LWW. Real products (Figma included) drop down to a sequence CRDT for text specifically. |
| **Wall-clock is not involved** | "Last writer" means last in *Lamport* order, which can disagree with what the user saw happen. Usually invisible; occasionally baffling. |

> **Figma's actual answer** is a hybrid, and worth knowing: a custom tree where the *server*
> is the tie-breaker (so, not a pure CRDT), because they judged CRDT metadata too expensive
> for their document sizes. That is a legitimate different answer to the same trade-off —
> being able to name *why* someone else chose differently is what separates a good interview
> answer from a memorised one.

## 🌍 Real-world references

| Product | Sync approach | Notes |
|---|---|---|
| **Figma** | Custom CRDT-like tree, server-authoritative tie-breaking | [Blog: How Figma's multiplayer works](https://www.figma.com/blog/how-figmas-multiplayer-technology-works/) |
| **Miro** | Op-based with server reconciliation | Large boards → viewport streaming |
| **Excalidraw** | Server-broadcast + optimistic local state | [excalidraw.com](https://excalidraw.com) — OSS, great to read |
| **tldraw** | Uses `@tldraw/sync` on top of a CRDT store | OSS |
| **Google Docs / Sheets** | Operational Transform (OT) | Older but battle-tested approach |
| **Yjs / Automerge** | General-purpose CRDT libraries | Drop-in for many apps |

We'll implement a **tiny CRDT** in notebook 3 to see *why* this works.


## 🧭 What's next

- **Notebook 2** — the data model (`Shape`, `Op`, `Presence`) and HTTP / WebSocket APIs.
- **Notebook 3** — deep dive: Lamport clocks, LWW CRDT merge, naive-vs-CRDT concurrency demo, pub/sub fan-out, snapshots, and presence TTL.
